# Laboratory 05 — Impulse response and convolution

In this laboratory you will kick an oscillator once and get everything: its frequency, its
damping, its quality factor, and the entire resonance curve that module 02 needed a whole
frequency sweep to measure. Then you will build the response to arbitrary forces out of
nothing but kicks, and check it against an integrator that has never heard of a Green
function.

The final measurement is the point of the module: two completely different routes to $Q$
from one noisy record, each with an uncertainty, and a test of whether they agree.

Work through it in order. Where the notebook asks you to predict, write your prediction in
the cell provided **before** running the next cell.

## Model specification

| | |
|---|---|
| **System** | module 02's oscillator seen as a map from a force history to a displacement history |
| **Dynamics** | $m\ddot{x} + b\dot{x} + kx = F(t)$, solved as the convolution $x = G * F$ or by velocity Verlet |
| **Boundary** | at rest before the force begins — the causal condition that selects this Green function |
| **Ensemble** | one deterministic input–output pair per force; measurement noise is Gaussian, independent, seeded |
| **Ignored** | the microscopic origin of $b$, and every nonlinearity — superposition of kicks *is* linearity |
| **Valid when** | the response is linear, the force is bounded and starts at a finite time, sampling is well above $\omega_d$ |
| **Failure modes** | nonlinear springs; force content above Nyquist; naive deconvolution, which amplifies noise without limit |

All the physics lives in `wavelab.oscillators` and `wavelab.fourier` — open them and read
them.


In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# wavelab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy and matplotlib, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# wavelab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("wavelab", deps=False)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelab import fourier, measurement, oscillators
from wavelab.validation import seed_study

MASS = 0.5  # kg
STIFFNESS = 8.0  # N/m
DAMPING = 0.1  # kg/s  ->  Q = 20
OMEGA0 = oscillators.natural_frequency(MASS, STIFFNESS)
PERIOD = 2 * np.pi / OMEGA0
GAMMA = oscillators.damping_rate(MASS, DAMPING)
OMEGA_D = np.sqrt(OMEGA0**2 - GAMMA**2 / 4)
T_D = 2 * np.pi / OMEGA_D
TRUE_Q = oscillators.quality_factor(MASS, STIFFNESS, DAMPING)

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

print(f"omega0 = {OMEGA0} rad/s   omega_d = {OMEGA_D:.6f} rad/s   Q = {TRUE_Q}")
print(f"the ringing outlives the kick by 2/gamma = {2 / GAMMA:.1f} s"
      f" = {2 / GAMMA / PERIOD:.1f} periods")


## Part 1 — One kick

`oscillators.impulse_response` is $G(t)$: the motion after a *unit* impulse, and nothing
more than module 01's free decay started with $x = 0$ and $v = 1/m$. Look at what it does at
the origin — it starts at zero and leaves at slope $1/m$ — and at the envelope it lives
inside.


In [ ]:
t = np.linspace(0.0, 12 * PERIOD, 3000)
green = oscillators.impulse_response(t, MASS, STIFFNESS, DAMPING)
envelope = np.exp(-GAMMA * t / 2) / (MASS * OMEGA_D)

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(t, green, lw=1.2, label="G(t)")
ax.plot(t, envelope, ls="--", color="0.6", lw=0.9)
ax.plot(t, -envelope, ls="--", color="0.6", lw=0.9)
ax.axhline(0.0, color="0.85", lw=0.8)
ax.set_xlabel("t (s)")
ax.set_ylabel("G  (m per N s)")
ax.set_title("the impulse response, inside its decaying envelope")
plt.tight_layout()
plt.show()

slope = (green[1] - green[0]) / (t[1] - t[0])
print(f"G(0)          = {green[0]:.3e}      (a kick moves nothing instantly)")
print(f"slope at t=0  = {slope:.4f}     vs 1/m = {1 / MASS}")
print(f"peak of G     = {green.max():.4f}     vs 1/(m omega_d) = {1 / (MASS * OMEGA_D):.4f}")

# Read omega_d straight off the record the way an experimenter would, from the crossings.
measured_omega_d = measurement.frequency_from_zero_crossings(t, green)
print(f"omega_d from the crossings = {measured_omega_d:.4f}  vs {OMEGA_D:.4f}")


## Part 2 — Predict

Before running the next cell, commit to answers.

1. You will now transform this single ringdown. What do you expect $|\hat{G}(\omega)|$ to
   look like — a spike, a bump, something else? How wide?
2. Module 02 measured a resonance curve by sweeping a drive across dozens of frequencies,
   waiting out the transient at each one. How close can one kick get to that curve?


**Your prediction:**

1.
2.


## Part 3 — The resonance curve, from one kick

Transform the ringdown and lay module 02's swept response on top of it. Nothing from the
sweep was used to make the ringdown, and nothing from the ringdown was used to make the
sweep.

One detail matters and is not cosmetic. Under this course's forward kernel $e^{-\ii\omega t}$,
$\hat{G}$ carries $+\ii\gamma\omega$ where module 02's phasor $X$ carries $-\ii\gamma\omega$,
so the two are **conjugates**. The cell measures what happens if you forget that.


In [ ]:
DT = 0.01
N = 2**15
centred = (np.arange(N) - N // 2) * DT  # the grid `fourier.spectrum` assumes: t = 0 in the middle
kick = oscillators.impulse_response(centred, MASS, STIFFNESS, DAMPING)
omega, kick_spectrum = fourier.spectrum(kick, DT)

band = (omega >= 0) & (omega < 3 * OMEGA0)
swept = oscillators.steady_state_response(omega[band], MASS, STIFFNESS, DAMPING, 1.0)

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.4))
left.plot(omega[band], np.abs(kick_spectrum[band]), lw=2.4, color="0.75", label="from one kick")
left.plot(omega[band], np.abs(swept), lw=1.0, color="crimson", label="module 02 sweep")
left.set_xlabel("omega (rad/s)"), left.set_ylabel("|response|"), left.legend()
right.plot(omega[band], np.angle(kick_spectrum[band]), lw=2.4, color="0.75")
right.plot(omega[band], -np.angle(swept), lw=1.0, color="crimson")
right.set_xlabel("omega (rad/s)"), right.set_ylabel("phase of G-hat (rad)")
plt.tight_layout()
plt.show()

worst = np.max(np.abs(kick_spectrum[band] - np.conj(swept))) / np.max(np.abs(swept))
wrong = np.max(np.abs(kick_spectrum[band] - swept)) / np.max(np.abs(swept))
print(f"kick spectrum vs conjugated sweep: worst relative difference {worst:.2e}")
print(f"vs the sweep taken at face value:  {wrong:.2f}"
      "   <- forgetting the conjugate is a 200% error")


## Part 4 — Kick trains

Two kicks, and the spacing decides everything. In phase they add, in antiphase they nearly
cancel — nearly, because the first has already decayed a little by the time the second
lands. The closed forms are $1 + e^{-\pi/Q}$ and $1 - e^{-\pi/2Q}$.


In [ ]:
alone = oscillators.impulse_response(t, MASS, STIFFNESS, DAMPING)
settled = t > 3 * T_D
reference = np.max(np.abs(alone[settled]))

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(t, alone, lw=0.9, color="0.7", label="one kick")

print("  spacing              measured   predicted")
for name, spacing, predicted in (
    ("one period", T_D, 1 + np.exp(-np.pi / TRUE_Q)),
    ("half a period", T_D / 2, abs(1 - np.exp(-np.pi / (2 * TRUE_Q)))),
    ("a quarter period", T_D / 4, np.sqrt(2.0)),
):
    pair = alone + oscillators.impulse_response(t - spacing, MASS, STIFFNESS, DAMPING)
    late = t > spacing + 3 * T_D
    ratio = np.max(np.abs(pair[late])) / reference
    ax.plot(t, pair, lw=1.2, label=name)
    print(f"  {name:18s}   {ratio:6.3f}     {predicted:6.3f}")

ax.set_xlabel("t (s)"), ax.set_ylabel("x (m)"), ax.legend(fontsize=8)
ax.set_title("two kicks, three spacings")
plt.tight_layout()
plt.show()


## Part 5 — Any force at all

`convolution_response` adds up kicks. `simulate_forced` steps the differential equation
forward in time and knows nothing about Green functions. If the theory is right they must
agree on every force you can think of.


In [ ]:
dt = PERIOD / 400
n = 4000
grid = np.arange(n) * dt
forces = {
    "step": np.ones(n),
    "ramp": np.clip(grid / (4 * PERIOD), 0.0, 1.0),
    "burst": np.where(grid < 5 * PERIOD, np.cos(OMEGA0 * grid), 0.0),
    "noise": rng.normal(0.0, 1.0, n),
}

fig, axes = plt.subplots(2, 2, figsize=(10, 5), sharex=True)
worst_gap = 0.0
for axis, (name, force) in zip(axes.ravel(), forces.items(), strict=True):
    convolved = oscillators.convolution_response(force, dt, MASS, STIFFNESS, DAMPING)
    stepped = oscillators.simulate_forced(MASS, STIFFNESS, 0.0, 0.0, dt, force, damping=DAMPING)
    gap = float(np.max(np.abs(convolved - stepped.positions)))
    worst_gap = max(worst_gap, gap / np.max(np.abs(convolved)))
    axis.plot(grid, convolved, lw=2.2, color="0.75")
    axis.plot(stepped.times, stepped.positions, lw=0.8, color="crimson")
    axis.set_title(f"{name}: max gap {gap:.1e} m", fontsize=9)
    axis.set_ylabel("x (m)")
    print(f"  {name:6s}: max |convolution - Verlet| = {gap:.2e} m"
          f"   (peak {np.max(np.abs(convolved)):.3f} m)")
axes[1, 0].set_xlabel("t (s)"), axes[1, 1].set_xlabel("t (s)")
plt.tight_layout()
plt.show()


## Part 6 — A featureless force, a ringing answer

The step force has exactly one feature in its whole history. Watch what the oscillator does
with it, and how the overshoot depends on $Q$ alone — the closed form is
$\exp(-\pi\gamma/2\omega_d)$.


In [ ]:
ts = np.linspace(0.0, 30 * PERIOD, 6000)
critical = 2 * np.sqrt(MASS * STIFFNESS)

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.axhline(1.0, color="0.8", ls="--", lw=1.0)
print("     Q   overshoot   closed form")
for damping in (critical, 2.0, 1.0, 0.4, 0.1):
    q = oscillators.quality_factor(MASS, STIFFNESS, damping)
    response = oscillators.step_response(ts, MASS, STIFFNESS, damping, 1.0) * STIFFNESS
    gamma_d = oscillators.damping_rate(MASS, damping)
    squared = OMEGA0**2 - gamma_d**2 / 4
    predicted = np.exp(-np.pi * gamma_d / (2 * np.sqrt(squared))) if squared > 0 else 0.0
    ax.plot(ts, response, lw=1.3, label=f"Q = {q:.2f}")
    print(f"  {q:5.2f}   {100 * (response.max() - 1):6.2f}%     {100 * predicted:6.2f}%")

ax.set_xlabel("t (s)"), ax.set_ylabel("x / (F0/k)"), ax.legend(fontsize=8)
ax.set_title("step response: the force has one feature, the motion has dozens")
plt.tight_layout()
plt.show()


## Part 7 — Two routes to $Q$, with an error bar

This is the measurement the module is built around. One noisy ringdown, read two ways:

- **the envelope**, in the time domain, with `q_from_ringdown`;
- **the linewidth**, in the frequency domain, with `q_from_linewidth`.

Nothing passes between them but the data. Eight independent seeds give each an uncertainty,
and then the question is whether they agree within it.


In [ ]:
NOISE = 2e-4 * float(np.max(np.abs(kick)))


def q_from_envelope(generator):
    """Time domain: how fast does the ringing die away?"""
    noisy = measurement.add_noise(kick, NOISE, generator)
    return oscillators.q_from_ringdown(centred[N // 2:], noisy[N // 2:])


def q_from_line(generator):
    """Frequency domain: how wide is the line? Fitted, not read off two crossings — see part 8."""
    noisy = measurement.add_noise(kick, NOISE, generator)
    w, spec = fourier.spectrum(noisy, DT)
    keep = (w > OMEGA0 - 4 * GAMMA) & (w < OMEGA0 + 4 * GAMMA)
    return oscillators.q_from_linewidth(w[keep], np.abs(spec[keep]))


ring = seed_study(q_from_envelope, n_seeds=8, base_seed=7)
line = seed_study(q_from_line, n_seeds=8, base_seed=7)

print(f"  Q from the envelope   = {ring.mean:.4f} +/- {ring.standard_error:.4f}")
print(f"  Q from the linewidth  = {line.mean:.4f} +/- {line.standard_error:.4f}")
print(f"  true Q                = {TRUE_Q}")
print(f"  the two routes differ by {abs(ring.mean - line.mean):.4f}, "
      f"or {100 * abs(ring.mean - line.mean) / TRUE_Q:.2f}% of Q")
print()
print("  Note how small the seed-to-seed scatter is compared with the offset from the true")
print("  value. Both routes carry a systematic larger than their noise, and telling those")
print("  two kinds of error apart is the whole skill.")


## Part 8 — The trap

`q_from_bandwidth` takes exactly the same arguments as `q_from_linewidth` and answers the
same question by interpolating the two half-power crossings. On a *swept* curve that is the
right thing to do. On the spectrum of a ringdown it is a trap, and it will not warn you.

The reason is arithmetic, not noise. A record of duration $T$ has bins spaced $2\pi/T$, and
the line it must resolve has width $\gamma$. A record holding $n$ amplitude e-foldings has
$T = 2n/\gamma$, so the bins across the full width number $n/\pi$ — **independent of $Q$**.


In [ ]:
# A record holding three amplitude e-foldings: realistic, since after that the ringing has
# fallen to 5% of where it started and is heading for the noise floor.
short_n = 2 * int((3 * 2 / GAMMA) / DT)
short_t = (np.arange(short_n) - short_n // 2) * DT
short_kick = oscillators.impulse_response(short_t, MASS, STIFFNESS, DAMPING)
w_s, spec_s = fourier.spectrum(short_kick, DT)
keep_s = (w_s > 0.5 * OMEGA0) & (w_s < 1.5 * OMEGA0)

bins = GAMMA / (w_s[1] - w_s[0])
crossings = oscillators.q_from_bandwidth(w_s[keep_s], np.abs(spec_s[keep_s]))
fitted = oscillators.q_from_linewidth(w_s[keep_s], np.abs(spec_s[keep_s]))

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(w_s[keep_s], np.abs(spec_s[keep_s]), marker="o", ms=3, lw=0.8)
ax.set_xlabel("omega (rad/s)"), ax.set_ylabel("|G-hat|")
ax.set_title(f"the line, as actually sampled: {bins:.2f} bins across its full width")
plt.tight_layout()
plt.show()

print(f"  bins across the full width : {bins:.2f}")
print(f"  q_from_bandwidth (crossings): {crossings:6.2f}"
      f"   <- {100 * (crossings / TRUE_Q - 1):+.0f}%")
print(f"  q_from_linewidth (fitted)   : {fitted:6.2f}   <- {100 * (fitted / TRUE_Q - 1):+.0f}%")
print(f"  true Q                      : {TRUE_Q:6.2f}")
print()
for e_folds in (3, 5, 10, 31):
    print(f"  {e_folds:2d} e-foldings -> {e_folds / np.pi:5.2f} bins across the width, "
          f"signal down to {np.exp(-e_folds):.1e} of its start")
print()
print("  Recording for longer is not the escape it looks like.")


## Part 9 — Undoing a convolution, and failing

If $\hat{x} = \hat{G}\hat{F}$ then $\hat{F} = \hat{x}/\hat{G}$, so a measured motion should
give back the force that caused it. Try it, and add a little noise.


In [ ]:
probe = np.where((centred > 0) & (centred < 6.0), 1.0, 0.0)  # a force we will try to recover
motion = fourier.convolve(kick, probe, DT)
_, green_spectrum = fourier.spectrum(kick, DT)
trusted = np.abs(green_spectrum) > 0.05 * np.max(np.abs(green_spectrum))

print("     noise      naive peak   truncated peak   (the true force peaks at 1.0)")
for level in (0.0, 1e-6, 1e-4, 1e-2):
    noisy_motion = motion + rng.normal(0.0, level * np.max(np.abs(motion)), N)
    _, noisy_spectrum = fourier.spectrum(noisy_motion, DT)
    naive = np.real(fourier.inverse_spectrum(noisy_spectrum / green_spectrum, DT))
    rescued = np.real(
        fourier.inverse_spectrum(np.where(trusted, noisy_spectrum / green_spectrum, 0.0), DT)
    )
    print(f"  {level:8.0e}   {np.max(np.abs(naive)):10.2e}   {np.max(np.abs(rescued)):10.3f}")

print()
print("  Dividing by G-hat multiplies by a large number wherever G-hat is small, and what")
print("  lives out there is noise. Refusing to divide below a threshold keeps the answer")
print("  bounded and throws away real information with the noise. Module 53 does better.")


In [ ]:
# 1. The kick spectrum reproduces module 02's swept response, once conjugated.
assert worst < 1e-4

# 2. And the conjugate is load-bearing: getting it wrong is not a small error.
assert wrong > 0.5

# 3. G starts at zero and leaves the origin at 1/m.
assert green[0] == 0.0
assert abs(slope - 1 / MASS) / (1 / MASS) < 1e-3

# 4. Causality: nothing moves before the kick.
before = oscillators.impulse_response(np.linspace(-5, -0.01, 50), MASS, STIFFNESS, DAMPING)
assert np.all(before == 0.0)

# 5. Convolution and Verlet agree on every force tried, to well under a percent of the peak.
assert worst_gap < 1e-2

# 6. Two routes to Q from one record, agreeing to better than 1%.
assert abs(ring.mean - line.mean) / TRUE_Q < 1e-2
assert abs(ring.mean - TRUE_Q) / TRUE_Q < 2e-2
assert abs(line.mean - TRUE_Q) / TRUE_Q < 2e-2

# 7. The trap is real: on an under-resolved line the crossing route reads low and the fit does not.
assert bins < 2.0
assert abs(crossings - TRUE_Q) / TRUE_Q > 0.15
assert abs(fitted - TRUE_Q) / TRUE_Q < 0.06

print("all checks passed")
print(f"one kick reproduced the swept resonance curve to {worst:.1e} relative")


## Part 10 — Explore it yourself

Place your own kicks and watch the response assemble itself out of them. Try two kicks at
the same spacing you measured in part 4; try a burst of ten at random times; try raising $Q$
until each kick's ringing survives to meet the next one.


In [ ]:
import ipywidgets as widgets


def explore(quality=20.0, n_kicks=4, spacing=1.0, seed=3):
    damping = np.sqrt(MASS * STIFFNESS) / quality
    gamma = oscillators.damping_rate(MASS, damping)
    omega_d = np.sqrt(OMEGA0**2 - gamma**2 / 4)
    period_d = 2 * np.pi / omega_d

    local = np.random.default_rng(seed)
    grid = np.linspace(0.0, max(14 * PERIOD, (n_kicks + 4) * spacing * period_d), 3000)
    times = np.arange(n_kicks) * spacing * period_d + 0.5 * PERIOD
    strengths = local.choice([-1.0, 1.0], n_kicks)

    parts = [s * oscillators.impulse_response(grid - t0, MASS, STIFFNESS, damping)
             for t0, s in zip(times, strengths, strict=True)]
    total = np.sum(parts, axis=0)

    fig, (top, bottom) = plt.subplots(2, 1, figsize=(10, 4), sharex=True,
                                      gridspec_kw={"height_ratios": [1, 2.4]})
    top.vlines(times, 0, strengths, color="darkorange", lw=1.6)
    top.axhline(0, color="0.85", lw=0.8), top.set_ylabel("F"), top.set_ylim(-1.3, 1.3)
    for part in parts:
        bottom.plot(grid, part, lw=0.8, color="0.75")
    bottom.plot(grid, total, lw=1.8)
    bottom.axhline(0, color="0.85", lw=0.8)
    bottom.set_xlabel("t (s)"), bottom.set_ylabel("x (m)")
    plt.tight_layout()
    plt.show()

    print(f"Q = {quality:.1f}   ringing lifetime 2/gamma = {2 / gamma:.1f} s = "
          f"{2 / gamma / period_d:.1f} periods   peak |x| = {np.max(np.abs(total)):.3f} m")


widgets.interact_manual(
    explore,
    quality=widgets.FloatSlider(min=1.0, max=40.0, step=0.5, value=20.0, description="Q"),
    n_kicks=widgets.IntSlider(min=1, max=8, step=1, value=4, description="kicks"),
    spacing=widgets.FloatSlider(min=0.25, max=3.0, step=0.05, value=1.0, description="spacing/T"),
    seed=widgets.IntSlider(min=0, max=20, step=1, value=3, description="seed"),
);


## Check your understanding


In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "05-impulse-response.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")


## Before you leave

Write a few sentences on each, in the cell below.

1. In part 3 you recovered a whole resonance curve from one kick. What did you *not* get
   that a real frequency sweep would have given you? Think about noise, and about what a
   sweep does that a single record cannot.
2. In part 8 the two estimators disagreed by 18% on identical data, and neither raised an
   error. What would have to change about `q_from_bandwidth` for it to refuse the job, and
   why might a library author reasonably decide not to make it refuse?
3. Part 9's deconvolution failed at 1% noise. Suppose you could design the input force
   yourself instead of being handed one. What force would make the recovery easiest, and
   what does your answer say about why engineers use hammers?


**Your answers:**

1.
2.
3.
